# UAV Training - YOLO Detection v0.7.0

Tek hucre calistirin, her sey otomatik.

In [ ]:
VERSION        = '0.7.0'
REPO_URL       = 'https://github.com/CELEBI-AIA/AIA-training.git'
REPO_BRANCH    = 'main'
DRIVE_DATASET  = '/content/drive/MyDrive/AIA/datasets.tar.gz'
LOCAL_CACHE    = '/content/datasets_local'
DRIVE_RUNS     = '/content/drive/MyDrive/AIA/runs'
DRIVE_UPLOAD   = '/content/drive/MyDrive/AIA'
TRAIN_SCRIPT   = 'uav_training/train.py'

import subprocess, sys, os, glob, time, gc
from datetime import datetime

os.environ['PYTHONUNBUFFERED'] = '1'

def _run(cmd, check=True, **kw):
    sys.stdout.flush(); sys.stderr.flush()
    r = subprocess.run(cmd, shell=True, check=check,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    if r.stdout:
        sys.stdout.write(r.stdout)
        sys.stdout.flush()
    return r

def _banner(msg):
    sep = '=' * 60
    print(f'\n{sep}\n  {msg}\n{sep}', flush=True)

print(f'\n\U0001f6f0\ufe0f  UAV Training Bootstrap v{VERSION}', flush=True)

# === 0. Pre-Flight Cleanup ===
_banner('0/8 -- Pre-flight cleanup')
_run("pkill -9 -f 'uav_training/train.py' 2>/dev/null || true", check=False)
_run("pkill -9 -f 'yolo' 2>/dev/null || true", check=False)
try:
    import torch
    if torch.cuda.is_available(): torch.cuda.empty_cache()
except: pass
gc.collect()
REPO_DIR = '/content/repo'
if os.path.isdir(REPO_DIR):
    _run(f'find {REPO_DIR} -type d -name __pycache__ -exec rm -rf {{}} + 2>/dev/null || true', check=False)
_run('rm -rf /tmp/pip-* /tmp/torch_* 2>/dev/null || true', check=False)
print('  Done', flush=True)

# === 1. Mount Google Drive ===
_banner('1/8 -- Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
if not os.path.isfile(DRIVE_DATASET):
    raise FileNotFoundError(f'Dataset not found: {DRIVE_DATASET}')
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(DRIVE_UPLOAD, exist_ok=True)
tar_size = os.path.getsize(DRIVE_DATASET) / (1024**3)
print(f'  Archive: {tar_size:.2f} GB', flush=True)

# === 2. Repository ===
_banner('2/8 -- Setting up repository')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    _run(f'git -C {REPO_DIR} fetch --all')
    _run(f'git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}')
else:
    _run(f'git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
_run(f'git -C {REPO_DIR} log --oneline -1')

# === 3. Dependencies ===
_banner('3/8 -- Installing dependencies')
req = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.isfile(req):
    _run(f'{sys.executable} -m pip install --progress-bar on -r {req}')
_run(f'{sys.executable} -m pip install --progress-bar on ultralytics psutil', check=False)

# === 4. Hardware Detection (BEFORE dataset download!) ===
_banner('4/8 -- Hardware detection & GPU check')
# Check GPU/TPU FIRST so we don't waste time downloading 78GB on a TPU
_is_tpu = os.environ.get('TPU_NAME') is not None or os.environ.get('COLAB_TPU_ADDR') is not None or os.path.exists('/dev/accel0')
try:
    import torch as _t
    _has_cuda = _t.cuda.is_available()
    if _has_cuda:
        _p = _t.cuda.get_device_properties(0)
        print(f'  GPU: {_p.name}  |  VRAM: {_p.total_memory / 1024**3:.1f} GB', flush=True)
        del _p
    del _t
except Exception:
    _has_cuda = False
if _is_tpu and not _has_cuda:
    raise RuntimeError('TPU detected! YOLO requires CUDA GPU. Change to: Runtime > Change runtime type > GPU')
if not _has_cuda:
    print('  WARNING: No CUDA GPU found!', flush=True)
_run('nvidia-smi', check=False)
os.environ['UAV_PROJECT_DIR'] = DRIVE_RUNS
os.environ['DRIVE_UPLOAD_DIR'] = DRIVE_UPLOAD
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENCV_FOR_THREADS_NUM'] = '1'
_run('yolo settings runs_dir="/content/runs"', check=False)
_run('yolo settings datasets_dir="/content/datasets_local"', check=False)
print('  All settings configured', flush=True)

# === 5. Dataset: Download -> Extract -> Verify ===
_banner('5/8 -- Dataset to local SSD')
_run('apt-get update -qq && apt-get install -y pigz pv 2>&1 | tail -3', check=False)
os.makedirs(LOCAL_CACHE, exist_ok=True)
NCPU = os.cpu_count() or 2
MARKER = os.path.join(LOCAL_CACHE, '.done')
LOCAL_TAR = '/content/datasets.tar.gz'
existing = sum(len(f) for _,_,f in os.walk(LOCAL_CACHE))
t0 = time.time()
if os.path.isfile(MARKER):
    print(f'  Cache complete ({existing} files) -- SKIP', flush=True)
elif existing > 5000:
    print(f'  Dataset detected ({existing} files) -- SKIP', flush=True)
    open(MARKER, 'w').write('1')
else:
    tar_gb = os.path.getsize(DRIVE_DATASET) / (1024**3)
    if os.path.isfile(LOCAL_TAR) and os.path.getsize(LOCAL_TAR) == os.path.getsize(DRIVE_DATASET):
        print(f'  tar.gz on SSD ({tar_gb:.1f} GB) -- skip download', flush=True)
    else:
        if os.path.isfile(LOCAL_TAR): os.remove(LOCAL_TAR)
        print(f'  Downloading {tar_gb:.1f} GB to local SSD...', flush=True)
        _run(f'dd if="{DRIVE_DATASET}" of="{LOCAL_TAR}" bs=64M status=progress 2>&1 | tail -5')
    dl_t = time.time() - t0
    print(f'  Download: {dl_t:.0f}s ({tar_gb/max(dl_t,1)*1024:.0f} MB/s)', flush=True)
    t1 = time.time()
    TAR_FAST = '--no-same-owner --no-same-permissions -b 512'
    if existing > 100:
        _run(f'pigz -d -c -p {NCPU} "{LOCAL_TAR}" | tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} --skip-old-files --checkpoint=10000 --checkpoint-action=echo="  %u checked"')
    else:
        print(f'  Extracting (pigz {NCPU} cores)...', flush=True)
        _run(f'pigz -d -c -p {NCPU} "{LOCAL_TAR}" | tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} --checkpoint=10000 --checkpoint-action=echo="  %u extracted"')
    fc = sum(len(f) for _,_,f in os.walk(LOCAL_CACHE))
    print(f'  Extracted in {time.time()-t1:.0f}s -- {fc} files', flush=True)
    if fc > 5000:
        open(MARKER, 'w').write('1')
        if os.path.isfile(LOCAL_TAR):
            os.remove(LOCAL_TAR)
            print(f'  Deleted tar.gz -- freed {tar_gb:.1f} GB', flush=True)
    else:
        print(f'  WARNING: only {fc} files', flush=True)
print(f'  Total: {time.time()-t0:.0f}s', flush=True)
repo_ds = os.path.join(REPO_DIR, 'datasets')
if os.path.islink(repo_ds) or os.path.isdir(repo_ds):
    _run(f'rm -rf "{repo_ds}"')
os.symlink(LOCAL_CACHE, repo_ds)
_run('df -h /content | tail -1')

# === 6. Training ===
_banner('6/8 -- Starting training')
def find_ckpt(*dirs):
    for d in dirs:
        c = glob.glob(os.path.join(d, '**', 'weights', 'last.pt'), recursive=True)
        if c: return max(c, key=os.path.getmtime)
    return None
ckpt = find_ckpt('/content/runs', DRIVE_RUNS)
script = os.path.join(REPO_DIR, TRAIN_SCRIPT)
if not os.path.isfile(script):
    raise FileNotFoundError(f'Not found: {script}')
log_dir = '/content/logs'
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, datetime.now().strftime('log_%Y-%m-%d_%H-%M.txt'))
if ckpt:
    print(f'  Resuming: {ckpt}', flush=True)
    cmd = f'{sys.executable} -u "{script}" --resume'
else:
    print('  Fresh training', flush=True)
    cmd = f'{sys.executable} -u "{script}"'
os.chdir(REPO_DIR)
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        env={**os.environ, 'PYTHONUNBUFFERED': '1'})
with open(log_path, 'wb') as lf:
    fd = proc.stdout.fileno()
    while True:
        data = os.read(fd, 8192)
        if not data: break
        sys.stdout.write(data.decode('utf-8', errors='replace'))
        sys.stdout.flush()
        lf.write(data); lf.flush()
ec = proc.wait()
print(f'\n  Exit code: {ec}', flush=True)

# === 7. Sync to Drive ===
_banner('7/8 -- Syncing to Google Drive')
drive_log_dir = os.path.join(DRIVE_UPLOAD, 'logs')
os.makedirs(drive_log_dir, exist_ok=True)
_run(f'rsync -a "{log_dir}/" "{drive_log_dir}/"', check=False)
if os.path.exists('/content/runs'):
    _run(f'rsync -a "/content/runs/" "{DRIVE_RUNS}/"', check=False)

# === 8. Summary ===
_banner('8/8 -- Summary')
models_dir = os.path.join(DRIVE_UPLOAD, 'models')
if os.path.isdir(models_dir):
    for mf in sorted(os.listdir(models_dir))[-5:]:
        if os.path.isdir(os.path.join(models_dir, mf)):
            print(f'  {mf} ({len(os.listdir(os.path.join(models_dir, mf)))} files)')
for bf in sorted(glob.glob(os.path.join(DRIVE_UPLOAD, 'best_*.pt'))):
    print(f'  -> {os.path.basename(bf)} ({os.path.getsize(bf)/1024**2:.1f} MB)')
_banner('Done -- all outputs on Google Drive')